# Exercise 6-03: NVIDIA Model Card++ & NIST AI RMF Governance Mapping

**Chapter 6: Trustworthy AI: Safety, Ethics, Privacy, and Governance**  
**Lab Type:** Interactive Worksheet Notebook | **Estimated Time:** 30–40 minutes  
**Exam Domains Covered:** Trustworthy AI (10%), Software Development (24%)

---

## Learning Objectives
By the end of this interactive worksheet, you will be able to:
1. Author a production **NVIDIA Model Card++** (including the four mandatory NVIDIA subcards: **Bias**, **Explainability**, **Privacy**, and **Safety & Security**) for the RAG Chatbot built in Lab 3-02 or the LoRA-tuned model built in Lab 4-01.
2. Map identified AI risks, metrics, and technical mitigations to the four core functions of the **NIST AI Risk Management Framework (AI RMF 1.0)**:
   - **GOVERN**, **MAP**, **MEASURE**, and **MANAGE**.
3. Classify AI systems under the **EU AI Act** risk tiers (**Unacceptable Risk**, **High-Risk**, **Limited/Transparency Risk**, **Minimal Risk**) and explain **SHAP** vs. **LIME** explainability.

---
## 1. Conceptual Foundation: NVIDIA Model Card++ & NIST AI RMF 1.0

In **Chapter 6 (*Documentation, Transparency, and Governance Frameworks*)**, the book highlights two essential governance structures tested on the NCA-GENL exam:

### 1. NVIDIA Model Card++ (Main Card + 4 Specialized Subcards)
Standard model cards describe architecture, intended use, training data, and benchmark metrics. **NVIDIA Model Card++** extends this with four dedicated trustworthiness subcards:
1. **Bias Subcard:** Subgroup evaluation results, counterfactual probes, demographic representation, and bias mitigations.
2. **Explainability Subcard:** How model predictions are interpreted—such as **SHAP** (game-theoretic Shapley values measuring global and local feature attribution), **LIME** (local linear surrogate model fit around a single perturbed prediction), attention visualization, or **RAG `[doc-id]` source citations**.
3. **Privacy Subcard:** PII scrubbing (`spaCy` / regex redaction), **Differential Privacy (DP-SGD)** noise injection, **Federated Learning**, and data retention/opt-out compliance (**GDPR** right to erasure).
4. **Safety & Security Subcard:** Red-team adversarial test results, prompt injection defenses, **NeMo Guardrails** configuration, and out-of-scope refusal bounds.

### 2. The Four Core Functions of NIST AI RMF 1.0 (High-Frequency Exam Topic!)

| NIST AI RMF Function | Core Question It Answers | Concrete Actions & Artifacts |
| :--- | :--- | :--- |
| **1. GOVERN** *(Cross-cutting)* | *Who is accountable, and what policies/processes govern AI across the organization?* | Establish AI Ethics Review Board, define acceptable-use policies, maintain model inventory, assign sign-off authority and incident escalation paths |
| **2. MAP** | *What is the deployment context, intended use, and what could go wrong for stakeholders?* | Document intended vs. out-of-scope uses, identify potential hallucinations, PII exposure, or allocational harms before deployment |
| **3. MEASURE** | *How do we quantitatively test, benchmark, and monitor those mapped risks?* | Run Lab 6-01 red-team suites (ASR/FP rate), Lab 6-02 counterfactual bias probes, RAG faithfulness/citation audits, and `lm-eval` benchmarks |
| **4. MANAGE** | *What technical controls, guardrails, and fallback procedures actively mitigate the measured risks?* | Deploy **NeMo Guardrails** (`config.yml` + Colang), enforce human-in-the-loop (HITL) review, trigger automated rollback on drift/safety alerts |

In [1]:
import json
import pandas as pd

# Complete an NVIDIA Model Card++ for our Lab 3-02 Enterprise RAG Assistant
MODEL_CARD_PLUS_PLUS = {
    "model_overview": {
        "name": "Enterprise-RAG-NIM-Assistant-v1.0",
        "base_llm": "meta/llama-3.1-8b-instruct (served via NVIDIA NIM)",
        "embedding_model": "nvidia/nv-embedqa-e5-v5 (FAISS Cosine Index)",
        "reranker_model": "nvidia/nv-rerankqa-mistral-4b-v3 (Top-10 -> Top-4)",
        "intended_use": "Internal technical support assistant answering grounded questions from curated engineering manuals with [doc-id] citations.",
        "out_of_scope_use": "Autonomous execution of production shell commands, legal/medical/financial advice, or HR hiring decisions.",
        "eu_ai_act_risk_tier": "Limited / Transparency Risk (requires clear disclosure that users are interacting with an AI system).",
    },
    "bias_subcard": {
        "known_risks": "Pretrained LLM weights may exhibit tone or length disparities across demographic names/pronouns.",
        "evaluation_protocol": "16-prompt counterfactual probe suite across 4 demographic groups (Lab 6-02).",
        "measured_metrics": "Post-mitigation Max Endorsement Gap = 0.00, Max Length Ratio = 1.00.",
        "mitigation": "Rubric-anchored system prompt + strict grounding in neutral technical documentation.",
    },
    "explainability_subcard": {
        "method": "Mandatory inline [doc-id] chunk citations + retrieval/reranker relevance score logging.",
        "local_vs_global_interpretability": "Every generated claim is traceable to the exact retrieved chunk IDs (#c0..#cN); feature-level classifiers audited with SHAP/LIME.",
    },
    "privacy_subcard": {
        "data_handling": "Zero customer PII is used for model weight updates; all queries pass through spaCy + regex PII redaction prior to embedding/LLM calls.",
        "compliance_controls": "Supports GDPR document deletion by removing vectors by doc_id from FAISS/Milvus index within 24 hours.",
    },
    "safety_and_security_subcard": {
        "adversarial_testing": "20-prompt Red-Team suite covering DAN jailbreaks, prompt injection, system prompt leaks, and off-topic politics/competitor queries (Lab 6-01).",
        "guardrail_controls": "NVIDIA NeMo Guardrails (Input self-check, Colang topical flows, Output citation & safety verification).",
        "red_team_results": "100% Attack Block Rate (0 Wrongly Allowed), 0% Over-Refusal Rate on benign technical support queries.",
    },
}

print("=== COMPLETED NVIDIA MODEL CARD++ (JSON SPECIFICATION) ===")
print(json.dumps(MODEL_CARD_PLUS_PLUS, indent=2))

=== COMPLETED NVIDIA MODEL CARD++ (JSON SPECIFICATION) ===
{
  "model_overview": {
    "name": "Enterprise-RAG-NIM-Assistant-v1.0",
    "base_llm": "meta/llama-3.1-8b-instruct (served via NVIDIA NIM)",
    "embedding_model": "nvidia/nv-embedqa-e5-v5 (FAISS Cosine Index)",
    "reranker_model": "nvidia/nv-rerankqa-mistral-4b-v3 (Top-10 -> Top-4)",
    "intended_use": "Internal technical support assistant answering grounded questions from curated engineering manuals with [doc-id] citations.",
    "out_of_scope_use": "Autonomous execution of production shell commands, legal/medical/financial advice, or HR hiring decisions.",
    "eu_ai_act_risk_tier": "Limited / Transparency Risk (requires clear disclosure that users are interacting with an AI system)."
  },
  "bias_subcard": {
    "known_risks": "Pretrained LLM weights may exhibit tone or length disparities across demographic names/pronouns.",
    "evaluation_protocol": "16-prompt counterfactual probe suite across 4 demographic groups (L

---
## 2. Mapping Risks & Controls to the Four NIST AI RMF Functions

Let's build a structured traceability matrix mapping our RAG & LoRA system controls directly to **GOVERN**, **MAP**, **MEASURE**, and **MANAGE**—and include an automated validator to verify how exam scenario questions map to each function!

In [2]:
nist_rmf_mapping = [
    {
        "NIST_Function": "GOVERN",
        "Core_Focus": "Policies, Accountability, Roles & Organizational Culture",
        "Lab_System_Artifact": "Establish AI Governance Charter requiring Model Card++ sign-off and defining acceptable vs. prohibited use cases before production launch.",
        "Exam_Keywords": "Policy, accountability, governance board, risk appetite, roles & responsibilities, inventory, compliance framework",
    },
    {
        "NIST_Function": "MAP",
        "Core_Focus": "Context, Intended Use, Stakeholder Impact & Risk Identification",
        "Lab_System_Artifact": "Identify that ungrounded RAG answers could mislead on-call engineers and that support tickets may contain customer SSNs/PII.",
        "Exam_Keywords": "Identify risks, intended use context, stakeholder impact, categorize system scope, map failure modes",
    },
    {
        "NIST_Function": "MEASURE",
        "Core_Focus": "Quantitative Testing, Red-Teaming, Bias Probes & Benchmarks",
        "Lab_System_Artifact": "Run Lab 6-01 Red-Team benchmark (FN/FP rates), Lab 6-02 Counterfactual Bias probe, and Lab 3-02 Citation Faithfulness checks.",
        "Exam_Keywords": "Evaluate, benchmark, red-team testing, calculate fairness metrics, monitor drift, counterfactual score",
    },
    {
        "NIST_Function": "MANAGE",
        "Core_Focus": "Active Mitigation, Guardrails, Incident Response & Rollback",
        "Lab_System_Artifact": "Deploy NeMo Guardrails Colang flows, enable spaCy PII scrubbing on input, and route low-confidence queries to human support engineers.",
        "Exam_Keywords": "Mitigate, deploy guardrails, block unsafe outputs, human-in-the-loop fallback, incident response, rollback",
    },
]

df_nist = pd.DataFrame(nist_rmf_mapping)
print("=== NIST AI RMF 1.0 TRACEABILITY MATRIX ===")
print(df_nist[["NIST_Function", "Core_Focus", "Lab_System_Artifact"]].to_string(index=False))

=== NIST AI RMF 1.0 TRACEABILITY MATRIX ===
NIST_Function                                                      Core_Focus                                                                                                                        Lab_System_Artifact
       GOVERN        Policies, Accountability, Roles & Organizational Culture Establish AI Governance Charter requiring Model Card++ sign-off and defining acceptable vs. prohibited use cases before production launch.
          MAP Context, Intended Use, Stakeholder Impact & Risk Identification               Identify that ungrounded RAG answers could mislead on-call engineers and that support tickets may contain customer SSNs/PII.
      MEASURE     Quantitative Testing, Red-Teaming, Bias Probes & Benchmarks              Run Lab 6-01 Red-Team benchmark (FN/FP rates), Lab 6-02 Counterfactual Bias probe, and Lab 3-02 Citation Faithfulness checks.
       MANAGE     Active Mitigation, Guardrails, Incident Response & Rollback     Deploy

---
## 3. Hands-On Exercise: Classify Exam Scenarios by NIST AI RMF Function & Explainability Tool

Run the self-check classifier below to test your ability to map real NCA-GENL exam scenarios to **GOVERN / MAP / MEASURE / MANAGE**, **EU AI Act Risk Tiers**, and **SHAP vs. LIME**.

In [3]:
# Interactive Self-Check Quiz & Reference Solutions for Chapter 6 Governance & Explainability
chapter6_scenarios = [
    (
        "An enterprise creates an executive AI review committee and mandates that every LLM deployment must have a designated risk owner.",
        "NIST AI RMF: GOVERN",
        "Sets organizational policy, accountability structure, and oversight roles."
    ),
    (
        "Before writing code, a product team documents the target users of a medical triage bot and catalogs potential harms if the bot hallucinates drug dosages.",
        "NIST AI RMF: MAP",
        "Establishes operational context, stakeholder impact, and risk identification."
    ),
    (
        "A QA engineer runs 500 counterfactual prompt pairs and a 100-item jailbreak red-team suite to quantify refusal and disparity rates.",
        "NIST AI RMF: MEASURE",
        "Quantitatively tests and benchmarks the identified risks."
    ),
    (
        "After discovering indirect prompt injection risks, the team enables NeMo Guardrails input/output rails and adds a human escalation button.",
        "NIST AI RMF: MANAGE",
        "Actively treats, mitigates, and responds to measured risks in deployment."
    ),
    (
        "To explain a single unexpected fraud flag to a customer, an analyst perturbs that specific transaction and fits a sparse linear model locally.",
        "Explainability: LIME (Local Interpretable Model-agnostic Explanations)",
        "Approximates a black-box model locally around a single instance using a simple interpretable surrogate."
    ),
    (
        "An ML engineer computes exact game-theoretic marginal feature contributions across all feature coalitions for both global and local explanations.",
        "Explainability: SHAP (SHapley Additive exPlanations)",
        "Uses cooperative game theory (Shapley values) with consistency and local accuracy guarantees."
    ),
]

print("=== CHAPTER 6 GOVERNANCE & EXPLAINABILITY SCENARIO CLASSIFIER ===\n")
for i, (scenario, answer, rationale) in enumerate(chapter6_scenarios, 1):
    print(f"Scenario {i}: {scenario}")
    print(f"  -> Classification : {answer}")
    print(f"  -> Why            : {rationale}\n")

=== CHAPTER 6 GOVERNANCE & EXPLAINABILITY SCENARIO CLASSIFIER ===

Scenario 1: An enterprise creates an executive AI review committee and mandates that every LLM deployment must have a designated risk owner.
  -> Classification : NIST AI RMF: GOVERN
  -> Why            : Sets organizational policy, accountability structure, and oversight roles.

Scenario 2: Before writing code, a product team documents the target users of a medical triage bot and catalogs potential harms if the bot hallucinates drug dosages.
  -> Classification : NIST AI RMF: MAP
  -> Why            : Establishes operational context, stakeholder impact, and risk identification.

Scenario 3: A QA engineer runs 500 counterfactual prompt pairs and a 100-item jailbreak red-team suite to quantify refusal and disparity rates.
  -> Classification : NIST AI RMF: MEASURE
  -> Why            : Quantitatively tests and benchmarks the identified risks.

Scenario 4: After discovering indirect prompt injection risks, the team enable